# Sample dataset for "Exploring Discover's search methods"

Supporting notebook for the article "Exploring Discover's search methods: data views, filters, KQL, Lucene, and ES|QL". It creates the two indices the article searches in Discover:

- `o11y-labs-discover-service-metrics`: 15-minute service metrics for 4 services across 3 regions, with a p95 latency incident on `checkout-api` in `us-central1`.
- `o11y-labs-service-catalog-lookup`: a 4-document service catalog used by the `LOOKUP JOIN` example (requires Elasticsearch 9.1+).

## Setup

You need a `.env` file with `ELASTICSEARCH_URL` and `ELASTICSEARCH_API_KEY`.

In [1]:
%pip install -q elasticsearch==9.1 python-dotenv==1.0


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import random
from datetime import datetime, timedelta, timezone

from dotenv import load_dotenv
from elasticsearch import Elasticsearch, helpers

load_dotenv()

es_client = Elasticsearch(
    os.getenv("ELASTICSEARCH_URL"), api_key=os.getenv("ELASTICSEARCH_API_KEY")
)
es_client.info()["version"]["number"]

'9.4.2'

## Create the metrics index

Keyword fields for the service dimensions, numeric fields for the measurements.

In [3]:
METRICS_INDEX = "o11y-labs-discover-service-metrics"

if es_client.indices.exists(index=METRICS_INDEX):
    es_client.indices.delete(index=METRICS_INDEX)

es_client.indices.create(
    index=METRICS_INDEX,
    mappings={
        "properties": {
            "@timestamp": {"type": "date"},
            "service": {
                "properties": {
                    "name": {"type": "keyword"},
                    "environment": {"type": "keyword"},
                    "version": {"type": "keyword"},
                }
            },
            "cloud": {"properties": {"region": {"type": "keyword"}}},
            "host": {"properties": {"name": {"type": "keyword"}}},
            "metrics": {
                "properties": {
                    "latency": {"properties": {"p95_ms": {"type": "float"}}},
                    "cpu": {"properties": {"pct": {"type": "float"}}},
                    "error": {"properties": {"rate": {"type": "float"}}},
                }
            },
        }
    },
)
print(f"Created {METRICS_INDEX}")

Created o11y-labs-discover-service-metrics


## Generate the metric documents

One document per service, region, and 15-minute interval on June 30, 2026, 14:00-20:00 UTC. `checkout-api` in `us-central1` (production) reports p95 latency in the 550-600 ms range from 15:45 to 18:45 UTC; everything else stays at its baseline.

In [4]:
SERVICES = ["checkout-api", "checkout-worker", "payments-api", "inventory-api"]
REGIONS = ["us-central1", "us-east4", "europe-west1"]
BASELINE_LATENCY = {  # (low, high) p95 ms per service
    "checkout-api": (180, 320),
    "checkout-worker": (250, 420),
    "payments-api": (150, 300),
    "inventory-api": (120, 260),
}

START = datetime(2026, 6, 30, 14, 0, tzinfo=timezone.utc)
SAMPLES = 24  # 15-minute steps: 14:00Z .. 19:45Z
INCIDENT_START = datetime(2026, 6, 30, 15, 45, tzinfo=timezone.utc)
INCIDENT_END = datetime(2026, 6, 30, 18, 45, tzinfo=timezone.utc)

random.seed(42)


def make_doc(ts, service, region, environment):
    incident = (
        service == "checkout-api"
        and region == "us-central1"
        and environment == "production"
        and INCIDENT_START <= ts <= INCIDENT_END
    )
    lo, hi = BASELINE_LATENCY[service]
    if incident:
        latency = round(random.uniform(550, 600), 1)
        cpu = round(random.uniform(0.70, 0.92), 2)
        error_rate = round(random.uniform(0.025, 0.06), 3)
    else:
        latency = round(random.uniform(lo, hi), 1)
        cpu = round(random.uniform(0.15, 0.45), 2)
        error_rate = round(random.uniform(0.0, 0.015), 3)
    version = "2026.06.30-1" if service == "checkout-api" else "2026.06.29-7"
    return {
        "@timestamp": ts.strftime("%Y-%m-%dT%H:%M:%S.000Z"),
        "service": {"name": service, "environment": environment, "version": version},
        "cloud": {"region": region},
        "host": {"name": f"{service}-{region}-01"},
        "metrics": {
            "latency": {"p95_ms": latency},
            "cpu": {"pct": cpu},
            "error": {"rate": error_rate},
        },
    }


docs = []
for i in range(SAMPLES):
    ts = START + i * timedelta(minutes=15)
    for service in SERVICES:
        for region in REGIONS:
            docs.append(make_doc(ts, service, region, "production"))
    # a little staging data so the environment filter matters
    docs.append(make_doc(ts, "checkout-api", "us-central1", "staging"))
    docs.append(make_doc(ts, "payments-api", "us-central1", "staging"))

helpers.bulk(
    es_client,
    ({"_index": METRICS_INDEX, "_source": d} for d in docs),
    refresh=True,
)
print(f"Indexed {len(docs)} metric documents")

Indexed 336 metric documents


## Create the service catalog lookup index

`index.mode: lookup` is what allows ES|QL to use this index on the right side of a `LOOKUP JOIN`.

In [5]:
LOOKUP_INDEX = "o11y-labs-service-catalog-lookup"

CATALOG = [
    {
        "service": {"name": "checkout-api"},
        "owner": {"team": "checkout-platform"},
        "slo": {"latency_target_ms": 350},
        "runbook": {"url": "https://runbooks.example.com/checkout-api/latency"},
    },
    {
        "service": {"name": "checkout-worker"},
        "owner": {"team": "checkout-platform"},
        "slo": {"latency_target_ms": 800},
        "runbook": {"url": "https://runbooks.example.com/checkout-worker/latency"},
    },
    {
        "service": {"name": "payments-api"},
        "owner": {"team": "payments-platform"},
        "slo": {"latency_target_ms": 400},
        "runbook": {"url": "https://runbooks.example.com/payments-api/latency"},
    },
    {
        "service": {"name": "inventory-api"},
        "owner": {"team": "inventory-core"},
        "slo": {"latency_target_ms": 600},
        "runbook": {"url": "https://runbooks.example.com/inventory-api/latency"},
    },
]

if es_client.indices.exists(index=LOOKUP_INDEX):
    es_client.indices.delete(index=LOOKUP_INDEX)

es_client.indices.create(
    index=LOOKUP_INDEX,
    settings={"index.mode": "lookup"},
    mappings={
        "properties": {
            "service": {"properties": {"name": {"type": "keyword"}}},
            "owner": {"properties": {"team": {"type": "keyword"}}},
            "slo": {"properties": {"latency_target_ms": {"type": "long"}}},
            "runbook": {"properties": {"url": {"type": "keyword"}}},
        }
    },
)

for entry in CATALOG:
    es_client.index(index=LOOKUP_INDEX, id=entry["service"]["name"], document=entry)
es_client.indices.refresh(index=LOOKUP_INDEX)
print(f"Indexed {len(CATALOG)} catalog documents into {LOOKUP_INDEX}")

Indexed 4 catalog documents into o11y-labs-service-catalog-lookup


## Verify

Run the article's final ES|QL query. It should return 13 rows, all `checkout-api` in `us-central1` above its 350 ms SLO target.

In [6]:
query = f"""
FROM {METRICS_INDEX}
| WHERE @timestamp >= "2026-06-30T15:00:00.000Z" AND @timestamp <= "2026-06-30T18:45:00.000Z"
| WHERE service.environment == "production"
| LOOKUP JOIN {LOOKUP_INDEX} ON service.name
| WHERE owner.team == "checkout-platform" AND metrics.latency.p95_ms > slo.latency_target_ms
| KEEP @timestamp, service.name, cloud.region, metrics.latency.p95_ms, slo.latency_target_ms, owner.team
| SORT @timestamp DESC
"""

resp = es_client.esql.query(query=query)
print(f"{len(resp['values'])} rows")
for row in resp["values"][:5]:
    print(row)

13 rows
['2026-06-30T18:45:00.000Z', 'checkout-api', 'us-central1', 575.4000244140625, 350, 'checkout-platform']
['2026-06-30T18:30:00.000Z', 'checkout-api', 'us-central1', 577.0999755859375, 350, 'checkout-platform']
['2026-06-30T18:15:00.000Z', 'checkout-api', 'us-central1', 574.7000122070312, 350, 'checkout-platform']
['2026-06-30T18:00:00.000Z', 'checkout-api', 'us-central1', 599.5, 350, 'checkout-platform']
['2026-06-30T17:45:00.000Z', 'checkout-api', 'us-central1', 598.7999877929688, 350, 'checkout-platform']


/var/folders/5w/qq_73hhn37v6k0qn0gqscwsw0000gn/T/ipykernel_92768/3405012694.py:11: ElasticsearchWarning: No limit defined, adding default limit of [1000]
  resp = es_client.esql.query(query=query)


## Cleanup

Run this only when you are done with the article.

In [7]:
for name in (METRICS_INDEX, LOOKUP_INDEX):
    if es_client.indices.exists(index=name):
        es_client.indices.delete(index=name)
print("Deleted indices")

Deleted indices
